# QLoRA Fine-Tuning for HTML → JSON Extraction

Fine-tune a small open model (**Qwen3-1.7B**) with **QLoRA** to turn product HTML
snippets into a fixed JSON object. QLoRA loads the frozen base model in 4-bit and trains
only small LoRA adapters, so the run fits on one modest GPU and finishes in minutes. The
result is exported as **GGUF** and served locally with Ollama.

The steps below: load the 500-example dataset → load the 4-bit base model → attach LoRA
adapters → supervised fine-tune → evaluate on held-out examples → try a prompt → export
to GGUF → serve it.


## Step 1 — Load the dataset

`json_extraction_dataset_500.json` must be in the working directory. It is a list of
500 objects, each with an `input` (the prompt plus HTML) and an `output` (the JSON we
want). The four keys are always `name`, `price`, `category` and `manufacturer`.


In [2]:
import json

file = json.load(open("json_extraction_dataset_500.json"))
print(f"Loaded {len(file)} examples")
print(json.dumps(file[1], indent=2))

Loaded 500 examples
{
  "input": "Extract the product information:\n<div class='product'><h2>iPad Air</h2><span class='price'>$1344</span><span class='category'>audio</span><span class='brand'>Dell</span></div>",
  "output": {
    "name": "iPad Air",
    "price": "$1344",
    "category": "audio",
    "manufacturer": "Dell"
  }
}


## Step 2 — Install dependencies

Pins `unsloth`, `transformers` and `trl`. Unsloth patches the training stack at import
time, so small version mismatches can break a run; the versions here are known to fit
together.


In [ ]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth  # Local / cloud setups
else:
    import torch
    v = re.match(r"[\d]{1,}\.[\d]{1,}", str(torch.__version__)).group(0)
    xformers = "xformers==" + {"2.10": "0.0.34", "2.9": "0.0.33.post1", "2.8": "0.0.32.post2"}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    !pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

## Step 3 — Check the environment

Fine-tuning needs a CUDA GPU. This cell fails loudly if one is not visible instead of
letting the run die later during model loading.


In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")

## Step 4 — Load the base model in 4-bit (QLoRA)

QLoRA starts by quantizing the frozen base model to 4-bit, which cuts its memory use
roughly 4×. Under the hood Unsloth uses 4-bit **NF4** quantization with double
quantization, equivalent to a `BitsAndBytesConfig` with `bnb_4bit_quant_type="nf4"` and
`bnb_4bit_use_double_quant=True`. Only the LoRA adapters added in step 6 are trained, and
those stay in higher precision. `get_chat_template` installs the exact ChatML template the
model expects, so training and inference use the *same* prompt format.


In [ ]:
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template

model_name = "unsloth/Qwen3-1.7B-unsloth-bnb-4bit"
max_seq_length = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=max_seq_length,
    load_in_4bit=True,  # QLoRA: quantize the frozen base model to 4-bit
)

# Use the model's native chat template so training and inference match.
tokenizer = get_chat_template(tokenizer, chat_template="qwen3-instruct")

## Step 5 — Build the training text and hold out an eval set

Every example becomes a three-message conversation (system → user → assistant) rendered
with the chat template, so the model learns the format we use later. The last 50 rows are
kept aside: step 9 measures the model on them, so the score reflects data it never
trained on.


In [ ]:
from datasets import Dataset

SYSTEM_PROMPT = (
    "You extract product information from HTML and reply with a single JSON object "
    'using the keys "name", "price", "category" and "manufacturer".'
)

def format_prompt(example):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": example["input"]},
        {"role": "assistant", "content": json.dumps(example["output"])},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False)

# Hold out the last 50 rows so step 9 can score untrained data.
train_rows, eval_rows = file[:450], file[450:]

dataset = Dataset.from_dict({"text": [format_prompt(row) for row in train_rows]})
print(f"Train: {len(train_rows)}  Eval: {len(eval_rows)}")
print(dataset[0]["text"])

## Step 6 — Attach LoRA adapters (the LoRA half of QLoRA)

Instead of updating all ~1.7B weights, LoRA freezes the 4-bit base model and trains small
low-rank matrices added to the attention and MLP layers. A 4-bit base plus LoRA adapters is
what makes fine-tuning fit a modest GPU and produce a small adapter file. `r=32` is the
rank (the width of those matrices); a higher rank can fit more, at more cost.


In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=32,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

## Step 7 — Configure the trainer

`SFTTrainer` runs supervised fine-tuning. The settings that matter:

| Setting | Value | Why |
|---|---|---|
| `per_device_train_batch_size` | 2 | Fits GPU memory |
| `gradient_accumulation_steps` | 4 | Effective batch size = 8 |
| `num_train_epochs` | 3 | Dataset is small (450 train rows) |
| `learning_rate` | 2e-4 | Typical QLoRA learning rate |
| `optim` | `adamw_8bit` | Memory-efficient optimizer |
| `lr_scheduler_type` | `linear` | Warm up, then decay to zero |

`train_on_responses_only` masks the prompt tokens, so the loss is computed **only on the
assistant's JSON**. Without it the model is also trained to reproduce the question.


In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    args=SFTConfig(
        dataset_text_field="text",
        max_length=max_seq_length,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,  # effective batch size = 8
        warmup_steps=10,
        num_train_epochs=3,
        learning_rate=2e-4,
        logging_steps=25,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        save_strategy="epoch",
        save_total_limit=2,
        output_dir="outputs",
        seed=3407,
        report_to="none",
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
    ),
)

# Only compute the loss on the assistant's JSON answer.
trainer = train_on_responses_only(trainer)

## Step 8 — Train

`trainer.train()` runs the loop and prints the wall-clock time and peak VRAM. On a 1.7B model this finishes in a few minutes.


In [ ]:
trainer_stats = trainer.train()
print(f"Training time: {trainer_stats.metrics['train_runtime'] / 60:.2f} minutes")
print(f"Peak VRAM: {torch.cuda.max_memory_reserved() / 1024**3:.2f} GB")

## Step 9 — Evaluate on held-out examples

Training loss says the model is fitting; it does not say it produces the right JSON. This
step runs the model over the 50 held-out rows and reports three numbers:

- **Valid JSON** — the reply parses as JSON at all.
- **Exact match** — all four keys match the target.
- **Per-field accuracy** — each key on its own, so you can see which field is weak.

Decoding is greedy (`do_sample=False`) so the score is deterministic and comparable
between runs. A high exact-match rate is the signal that the model learned the mapping, not
just the format.


In [ ]:
import json, re

FastLanguageModel.for_inference(model)

KEYS = ["name", "price", "category", "manufacturer"]

def predict(html):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": html},
    ]
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    ).to("cuda")
    out = model.generate(
        input_ids=inputs,
        max_new_tokens=64,
        do_sample=False,  # greedy, so the score is deterministic
    )
    return tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True).strip()

def parse_json(text):
    match = re.search(r"\{.*\}", text, re.DOTALL)
    if not match:
        return None
    try:
        return json.loads(match.group(0))
    except json.JSONDecodeError:
        return None

valid = exact = 0
field_hits = {key: 0 for key in KEYS}

for row in eval_rows:
    pred = parse_json(predict(row["input"]))
    gold = row["output"]
    if pred is None:
        continue
    valid += 1
    if all(pred.get(k) == gold[k] for k in KEYS):
        exact += 1
    for k in KEYS:
        field_hits[k] += pred.get(k) == gold[k]

n = len(eval_rows)
print(f"Held-out examples: {n}")
print(f"Valid JSON:        {valid / n:.1%}")
print(f"Exact match:       {exact / n:.1%}")
for k in KEYS:
    print(f"  {k:<13} {field_hits[k] / n:.1%}")


## Step 10 — Try one prompt

The same chat template as training, with `add_generation_prompt=True` so the model starts
writing the assistant turn. Qwen3-Instruct recommends `temperature=0.7, top_p=0.8,
top_k=20` for normal use; step 9 uses greedy decoding only because a benchmark should be
repeatable.


In [ ]:
from transformers import TextStreamer

FastLanguageModel.for_inference(model)

messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": "Extract the product information:\n<div class='product'><h2>iPad Air</h2><span class='price'>$1344</span><span class='category'>audio</span><span class='brand'>Dell</span></div>"},
]

inputs = tokenizer.apply_chat_template(
    messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
).to("cuda")

outputs = model.generate(
    input_ids=inputs,
    max_new_tokens=256,
    temperature=0.7,
    top_p=0.8,
    top_k=20,
    streamer=TextStreamer(tokenizer, skip_prompt=True),
)

print("\nFinal JSON:", tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True).strip())

## Step 11 — Export to GGUF

GGUF is the single-file format used by Ollama and llama.cpp. `q4_k_m` is a good
accuracy-vs-size trade-off (~1.1 GB for a 1.7B model). The file lands in
`gguf_model_gguf/`; the cell prints its exact path.


In [ ]:
# Export to GGUF (q4_k_m) for Ollama / llama.cpp.
model.save_pretrained_gguf("gguf_model", tokenizer, quantization_method="q4_k_m")

import glob
print("Exported:", glob.glob("gguf_model_gguf/*.gguf"))

## Step 12 — Serve it with Ollama

Copy the exported `.gguf` into this folder and make sure the `Modelfile` `FROM` line points
at it. Then either run Ollama by hand:

```bash
ollama create qwen3-json-extractor -f Modelfile
ollama run qwen3-json-extractor "Extract the product information:
<div class='product'><h2>iPad Air</h2><span class='price'>$1344</span><span class='category'>audio</span><span class='brand'>Dell</span></div>"
```

or use `./serve.sh`, which builds the Ollama model and opens a public
`trycloudflare.com` URL for it. See `README.md` for the full walkthrough.
